In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from io import BytesIO


PHENOTYPES_URL = (
    "https://raw.githubusercontent.com/"
    "neurohackademy/nh2020-curriculum/"
    "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b/"
    "tu-machine-learning-yarkoni/data/abide2_phenotypic.csv"
)

COLUMNS_URL = (
    "https://raw.githubusercontent.com/"
    "yoavmp/ml-neuro-tutorials/main/"
    "book/config/eda_phenotype_columns.json"
)

phenotypes = pd.read_csv(
    PHENOTYPES_URL,
    encoding="latin-1",
    low_memory=False,
)

phenotypes.columns = phenotypes.columns.str.strip()

selected_columns = pd.read_json(
    COLUMNS_URL,
    typ="series",
).tolist()

phenotypes = phenotypes[selected_columns].copy()

In [3]:
from io import BytesIO
import base64
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

histogram_variables = [
    "AGE_AT_SCAN",
    "FIQ",
    "VIQ",
    "PIQ",
    "SRS_TOTAL_RAW",
    "ADOS_G_TOTAL",
]

variable_selector = widgets.Dropdown(
    options=histogram_variables,
    value="AGE_AT_SCAN",
    description="Variable:",
    style={"description_width": "initial"},
)

bins_slider = widgets.IntSlider(
    value=25,
    min=5,
    max=100,
    step=5,
    description="Bins:",
    continuous_update=False,
    style={"description_width": "initial"},
)

status_text = widgets.HTML()
histogram_html = widgets.HTML()


def update_histogram(change=None):
    variable = variable_selector.value
    bins = bins_slider.value
    values = phenotypes[variable].dropna()

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.hist(
        values,
        bins=bins,
        color="steelblue",
        edgecolor="white",
    )

    ax.set(
        xlabel=variable,
        ylabel="Number of participants",
        title=(
            f"Distribution of {variable}\n"
            f"n = {len(values)}, "
            f"missing = {phenotypes[variable].isna().sum()}"
        ),
    )

    fig.tight_layout()

    image_buffer = BytesIO()
    fig.savefig(
        image_buffer,
        format="png",
        dpi=120,
        bbox_inches="tight",
    )
    plt.close(fig)

    encoded_image = base64.b64encode(
        image_buffer.getvalue()
    ).decode("ascii")

    status_text.value = (
        f"<b>Current selection:</b> {variable}, {bins} bins"
    )

    histogram_html.value = f"""
    <img
        src="data:image/png;base64,{encoded_image}"
        style="width:100%; max-width:800px;"
        alt="Histogram of {variable}"
    >
    """


variable_selector.observe(update_histogram, names="value")
bins_slider.observe(update_histogram, names="value")

# Create the initial plot before displaying the interface.
update_histogram()

display(
    widgets.VBox(
        [
            widgets.HBox(
                [variable_selector, bins_slider]
            ),
            status_text,
            histogram_html,
        ]
    )
)